# House Price Prediction – Boston Housing Dataset
**Models:** Linear Regression · Decision Tree Regression · Random Forest Regression

Each model is shown in two steps:
1. **Simple model** – only `RM` (rooms) is used, so we can draw the model as a line/curve on a graph.
2. **Full model** – all 13 features are used and the model is evaluated on test data.

Run the cells from top to bottom (**Run All**).

## 1. Import libraries and load data

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Loads data/boston.csv (or downloads it, or uses a synthetic copy if offline)
from src.data_loader import load_data
df, source = load_data()
print("Data source:", source)
print("Shape:", df.shape)
df.head()

Data source: Boston Housing (downloaded from OpenML)
Shape: (506, 14)


,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2


In [ ]:
# Basic information
print("Missing values:", df.isna().sum().sum())
df = df.fillna(df.median())      # fill any missing value with the column median
df.describe()

## 2. Quick look at the data

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(df["MEDV"], kde=True, color="steelblue", ax=ax[0])
ax[0].set_title("Distribution of House Price (MEDV)")

sns.heatmap(df.corr(), annot=True, fmt=".1f", cmap="coolwarm", annot_kws={"size": 7}, ax=ax[1])
ax[1].set_title("Correlation Heatmap")

plt.tight_layout()
plt.show()

## 3. Train / test split (used for the full models)

In [ ]:
X = df.drop("MEDV", axis=1)      # all 13 features
Y = df["MEDV"]                   # target: house price in $1000s

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
print("Training rows:", len(X_train), "| Testing rows:", len(X_test))

# 4. Linear Regression

### 4.1 Simple Linear Regression (RM → MEDV)

In [ ]:
# X = RM (number of rooms)
X_simple = df[["RM"]]
Y_simple = df["MEDV"]

# Train simple linear regression model
simple_model = LinearRegression()
simple_model.fit(X_simple, Y_simple)

# Create values for regression line
x_line = pd.DataFrame(np.linspace(X_simple["RM"].min(), X_simple["RM"].max(), 100), columns=["RM"])

# Predict values for the line
y_line = simple_model.predict(x_line)

# Plot actual data points
plt.figure(figsize=(7, 5))
plt.scatter(X_simple, Y_simple, color="blue", label="Actual Data")

# Plot regression line
plt.plot(x_line, y_line, color="red", linewidth=2, label="Regression Line")

plt.xlabel("Average Number of Rooms (RM)")
plt.ylabel("House Price (MEDV)")
plt.title("RM vs House Price - Linear Regression")
plt.legend()
plt.show()

print("Slope (price change per extra room): %.2f" % simple_model.coef_[0])
print("Intercept: %.2f" % simple_model.intercept_)

### 4.2 Multiple Linear Regression (all features)

In [ ]:
lin_model = LinearRegression()
lin_model.fit(X_train, Y_train)
lin_pred = lin_model.predict(X_test)

print("MAE  :", round(mean_absolute_error(Y_test, lin_pred), 3))
print("RMSE :", round(np.sqrt(mean_squared_error(Y_test, lin_pred)), 3))
print("R2   :", round(r2_score(Y_test, lin_pred), 3))

fig, ax = plt.subplots(1, 2, figsize=(13, 5))

# Actual vs predicted
ax[0].scatter(Y_test, lin_pred, color="blue")
ax[0].plot([Y.min(), Y.max()], [Y.min(), Y.max()], "r--", label="Perfect prediction")
ax[0].set_xlabel("Actual Price")
ax[0].set_ylabel("Predicted Price")
ax[0].set_title("Linear Regression: Actual vs Predicted")
ax[0].legend()

# Coefficients (effect of each feature)
coef = pd.Series(lin_model.coef_, index=X.columns).sort_values()
coef.plot(kind="barh", color="teal", ax=ax[1])
ax[1].set_title("Linear Regression Coefficients")

plt.tight_layout()
plt.show()

# 5. Decision Tree Regression

### 5.1 Simple Decision Tree (RM → MEDV)

In [ ]:
X_simple = df[["RM"]]
Y_simple = df["MEDV"]

# Train simple decision tree (max_depth=4 keeps the tree small)
tree_simple = DecisionTreeRegressor(max_depth=4, random_state=42)
tree_simple.fit(X_simple, Y_simple)

x_line = pd.DataFrame(np.linspace(X_simple["RM"].min(), X_simple["RM"].max(), 300), columns=["RM"])
y_line = tree_simple.predict(x_line)

plt.figure(figsize=(7, 5))
plt.scatter(X_simple, Y_simple, color="blue", label="Actual Data")
plt.plot(x_line, y_line, color="red", linewidth=2, label="Decision Tree Prediction")

plt.xlabel("Average Number of Rooms (RM)")
plt.ylabel("House Price (MEDV)")
plt.title("RM vs House Price - Decision Tree")
plt.legend()
plt.show()

### 5.2 Decision Tree with all features

In [ ]:
tree_model = DecisionTreeRegressor(max_depth=5, min_samples_leaf=5, random_state=42)
tree_model.fit(X_train, Y_train)
tree_pred = tree_model.predict(X_test)

print("MAE  :", round(mean_absolute_error(Y_test, tree_pred), 3))
print("RMSE :", round(np.sqrt(mean_squared_error(Y_test, tree_pred)), 3))
print("R2   :", round(r2_score(Y_test, tree_pred), 3))

fig, ax = plt.subplots(1, 2, figsize=(13, 5))

ax[0].scatter(Y_test, tree_pred, color="blue")
ax[0].plot([Y.min(), Y.max()], [Y.min(), Y.max()], "r--", label="Perfect prediction")
ax[0].set_xlabel("Actual Price")
ax[0].set_ylabel("Predicted Price")
ax[0].set_title("Decision Tree: Actual vs Predicted")
ax[0].legend()

imp = pd.Series(tree_model.feature_importances_, index=X.columns).sort_values()
imp.plot(kind="barh", color="orange", ax=ax[1])
ax[1].set_title("Decision Tree Feature Importance")

plt.tight_layout()
plt.show()

### 5.3 Visualise the Decision Tree itself

In [ ]:
# Shows the first 3 levels of the tree: each box is a question (e.g. RM <= 6.9)
plt.figure(figsize=(20, 9))
plot_tree(tree_model, feature_names=X.columns, filled=True, rounded=True, max_depth=3, fontsize=10)
plt.title("Decision Tree (top 3 levels)")
plt.show()

# 6. Random Forest Regression

### 6.1 Simple Random Forest (RM → MEDV)

In [ ]:
X_simple = df[["RM"]]
Y_simple = df["MEDV"]

# Train simple random forest (many trees averaged together)
forest_simple = RandomForestRegressor(n_estimators=100, max_depth=4, random_state=42)
forest_simple.fit(X_simple, Y_simple)

x_line = pd.DataFrame(np.linspace(X_simple["RM"].min(), X_simple["RM"].max(), 300), columns=["RM"])
y_line = forest_simple.predict(x_line)

plt.figure(figsize=(7, 5))
plt.scatter(X_simple, Y_simple, color="blue", label="Actual Data")
plt.plot(x_line, y_line, color="red", linewidth=2, label="Random Forest Prediction")

plt.xlabel("Average Number of Rooms (RM)")
plt.ylabel("House Price (MEDV)")
plt.title("RM vs House Price - Random Forest")
plt.legend()
plt.show()

### 6.2 Random Forest with all features

In [ ]:
forest_model = RandomForestRegressor(n_estimators=200, random_state=42)
forest_model.fit(X_train, Y_train)
forest_pred = forest_model.predict(X_test)

print("MAE  :", round(mean_absolute_error(Y_test, forest_pred), 3))
print("RMSE :", round(np.sqrt(mean_squared_error(Y_test, forest_pred)), 3))
print("R2   :", round(r2_score(Y_test, forest_pred), 3))

fig, ax = plt.subplots(1, 2, figsize=(13, 5))

ax[0].scatter(Y_test, forest_pred, color="blue")
ax[0].plot([Y.min(), Y.max()], [Y.min(), Y.max()], "r--", label="Perfect prediction")
ax[0].set_xlabel("Actual Price")
ax[0].set_ylabel("Predicted Price")
ax[0].set_title("Random Forest: Actual vs Predicted")
ax[0].legend()

imp = pd.Series(forest_model.feature_importances_, index=X.columns).sort_values()
imp.plot(kind="barh", color="green", ax=ax[1])
ax[1].set_title("Random Forest Feature Importance")

plt.tight_layout()
plt.show()

### 6.3 Visualise trees inside the Random Forest

In [ ]:
# A random forest is a collection of trees. Here are 2 of its 200 trees (top 3 levels).
fig, ax = plt.subplots(1, 2, figsize=(26, 9))
for i in range(2):
    plot_tree(forest_model.estimators_[i], feature_names=X.columns, filled=True,
              rounded=True, max_depth=3, fontsize=8, ax=ax[i])
    ax[i].set_title("Random Forest - Tree number %d" % (i + 1))
plt.show()

# 7. Compare all three models

In [ ]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Decision Tree", "Random Forest"],
    "MAE":  [mean_absolute_error(Y_test, p) for p in (lin_pred, tree_pred, forest_pred)],
    "RMSE": [np.sqrt(mean_squared_error(Y_test, p)) for p in (lin_pred, tree_pred, forest_pred)],
    "R2":   [r2_score(Y_test, p) for p in (lin_pred, tree_pred, forest_pred)],
}).round(3)
display(results)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(results["Model"], results["R2"], color=["gray", "orange", "green"])
ax[0].set_title("R2 Score (higher is better)")
ax[0].set_ylim(0, 1)
ax[1].bar(results["Model"], results["RMSE"], color=["gray", "orange", "green"])
ax[1].set_title("RMSE (lower is better)")
plt.tight_layout()
plt.show()

print("Best model:", results.loc[results["R2"].idxmax(), "Model"])

In [ ]:
# All three models on the same graph (RM only)
x_line = pd.DataFrame(np.linspace(df["RM"].min(), df["RM"].max(), 300), columns=["RM"])

plt.figure(figsize=(9, 6))
plt.scatter(df["RM"], df["MEDV"], color="lightblue", label="Actual Data")
plt.plot(x_line, simple_model.predict(x_line), color="red", linewidth=2, label="Linear Regression")
plt.plot(x_line, tree_simple.predict(x_line), color="orange", linewidth=2, label="Decision Tree")
plt.plot(x_line, forest_simple.predict(x_line), color="green", linewidth=2, label="Random Forest")
plt.xlabel("Average Number of Rooms (RM)")
plt.ylabel("House Price (MEDV)")
plt.title("RM vs House Price - All Three Models")
plt.legend()
plt.show()

## 8. Predict the price of a new house

In [ ]:
new_house = pd.DataFrame([{
    "CRIM": 0.2, "ZN": 0, "INDUS": 8, "CHAS": 0, "NOX": 0.5, "RM": 6.5,
    "AGE": 60, "DIS": 4, "RAD": 4, "TAX": 300, "PTRATIO": 18, "B": 390, "LSTAT": 10
}])

print("Linear Regression :", round(lin_model.predict(new_house)[0], 2), "(x $1000)")
print("Decision Tree     :", round(tree_model.predict(new_house)[0], 2), "(x $1000)")
print("Random Forest     :", round(forest_model.predict(new_house)[0], 2), "(x $1000)")

In [4]:
%pip install seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
